# HeatMatch — Demo

**Orientation-Aware Visualization and Comparison of Very Dense Saccade Patterns**

This notebook demonstrates the two main contributions of HeatMatch:
1. **OOI-coded heatmaps** — visualizing saccade orientation fields over a stimulus grid
2. **HeatMatch similarity** — pairwise comparison of saccade patterns across stimuli

> Long et al. (2026). *Proceedings of the ACM on Computer Graphics and Interactive Techniques*, 9(2), Article 18. https://doi.org/10.1145/3803539

---

**Data:** The preprocessed saccade dataset is not included in this repository. Download `data_anonymized.csv` from [osf.io/f2xhj](https://osf.io/f2xhj/) and place it in a `tests/` folder at the repo root before running this notebook.

## Parameters

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from heatmatch import make_reference_grid, make_orientation_field, Heatmap, compute_similarity

%matplotlib inline

CSV_PATH       = os.path.join('tests', 'data_anonymized.csv')
W, H           = 2880, 2160  # stimulus size in pixels
REF_NX         = 200          # reference grid resolution
REF_NY         = 200
SIGMA          = 50.0         # Gaussian bandwidth σ (pixels)
OOI            = 0.0          # Orientation of Interest in degrees: 0=east, 90=north, 180=west, 270=south
DENSITY_WEIGHT = 1.0          # opacity blend: 1 → density only, 0 → coherence only
T              = 0.5          # density–coherence trade-off for S_dir — Eq. (6)

## Compute Orientation Fields

Build the reference grid once, then compute an orientation field per stimulus.
The fields are cached in a dict and reused for both heatmap rendering and similarity scoring.

In [ ]:
df = pd.read_csv(CSV_PATH)
pts, xx, yy = make_reference_grid(W, H, REF_NX, REF_NY)
stimuli = sorted(df['STIMULUS'].unique())

fields = {}
for stim in tqdm(stimuli, desc='Computing fields'):
    dfg    = df[df['STIMULUS'] == stim]
    onset  = dfg[['SAC_ONSET_X',  'SAC_ONSET_Y']].to_numpy(np.float64)
    offset = dfg[['SAC_OFFSET_X', 'SAC_OFFSET_Y']].to_numpy(np.float64)
    fields[stim] = make_orientation_field(pts, onset, offset, sigma=SIGMA, grid_shape=yy.shape)

## OOI-Coded Heatmaps

Each cell shows the saccade orientation field for one stimulus.
Hue encodes mean orientation relative to the OOI; opacity encodes local confidence.

In [ ]:
NROWS, NCOLS = 3, 4
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(3 * NCOLS * W / H, 3 * NROWS))

for ax, stim in zip(axes.ravel(), stimuli):
    Heatmap(fields[stim], W, H).draw(ax, ooi=OOI, density_weight=DENSITY_WEIGHT)
    ax.set_title(stim, fontsize=11)

fig.tight_layout()
plt.show()

## Pairwise Similarity Matrix

HeatMatch similarity *S* = (*S*_loc + *S*_dir) / 2 for all pairs of stimuli.

In [ ]:
n = len(stimuli)
s_mat = np.zeros((n, n))

pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
for i, j in tqdm(pairs, desc='Computing similarity'):
    result = compute_similarity(fields[stimuli[i]], fields[stimuli[j]], t=T)
    s_mat[i, j] = s_mat[j, i] = (result.s_loc + result.s_dir) / 2.0
np.fill_diagonal(s_mat, 1.0)

short_labels = [s[:10] for s in stimuli]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(s_mat, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(n)); ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(short_labels, fontsize=8)
ax.set_title(f'HeatMatch similarity  (σ={SIGMA:.0f} px,  t={T})', fontsize=12)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()